# Run wopke_100 extraction on mapped papers

Reads markdown from `data/wopke_100/paper_output`.
Uses `src/experimentutils/papermap.py` (folder → GT `Study#`) and by default runs the **90 mapped** papers only.

Runs **direct LLM**, **static workflow**, and **MAS**. Results go to `outputs/{tag}/` as CSV. Re-run to **resume**: finished paper/method CSVs are skipped.

In [ ]:
# --- LLM (edit these; overrides .env for this run) ---
PROVIDER = "surf"
MODEL_NAME = "RedHatAI/gemma-4-31B-it-NVFP4"

# --- Paths ---
PAPER_INPUT_DIR = "data/wopke_100/paper_output"

# --- Experiment ---
STANDARD_KEY = "wopke_100"
METHODS = ["direct_llm", "static_workflow", "mas"]
SKIP_EXISTING = True  # resume 1→90; skip done methods; skip incomplete gaps behind later results
ONLY_MAPPED = True  # True = papermap folders only (90); False = all 100 folders
STUDY_IDS = None  # None = all selected papers; or GT Study# list e.g. [1, 2, 10]
SKIP_STUDY_IDS = []  # temporary: skip Study# 3 (Bulson 1997) — hung on SURF Qwen
MAS_TOPOLOGY = "pipeline"
SHOW_LOGS = True  # INFO logs from orchestrator / workflow / LLM calls
QUIET_HTTP = True  # keep httpx/httpcore/openai chatter down

In [ ]:
from __future__ import annotations

import logging
import os
import re
import sys
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore")

# Restore progress logs (previously silenced with logging.disable)
logging.disable(logging.NOTSET)
root = logging.getLogger()
root.handlers.clear()
logging.basicConfig(
    level=logging.INFO if SHOW_LOGS else logging.WARNING,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
for name in ("httpx", "httpcore", "openai", "urllib3", "httpx._client"):
    logging.getLogger(name).setLevel(logging.WARNING if QUIET_HTTP else logging.INFO)
    logging.getLogger(name).disabled = False
# Project loggers
for name in ("src", "src.orchestrator", "src.players", "src.static_workflow", "src.direct_llm_call"):
    logging.getLogger(name).setLevel(logging.INFO if SHOW_LOGS else logging.WARNING)

repo_root = Path.cwd().resolve()
for p in [repo_root, *repo_root.parents]:
    if (p / "src").is_dir() and (p / "outputs").is_dir():
        repo_root = p
        break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv

load_dotenv(repo_root / ".env")
os.environ["LLM_PROVIDER"] = PROVIDER
os.environ["LLM_MODEL"] = MODEL_NAME

import src.config as cfg

cfg.LLM_PROVIDER = PROVIDER
cfg.DEFAULT_MODEL = MODEL_NAME

import pandas as pd
from tqdm.auto import tqdm

from src.context import create_context
from src.core.schema_factory import SchemaFactory
from src.direct_llm_call import extract_meta_analysis
from src.experimentutils import (
    get_all_markdown_paths,
    get_paper_info_from_path,
    highlight_numbers_and_tables,
    read_paper_text,
)
from src.experimentutils.output_utils import DEFAULT_OUTPUT_DIR
from src.experimentutils.papermap import (
    FOLDER_TO_STUDY_ID,
    UNMAPPED_FOLDERS,
    study_id_for_folder,
)
from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.static_workflow import run_two_step_text_to_dataset

log = logging.getLogger("run_wopke_100")

standard = METADATA_STANDARDS[STANDARD_KEY]
n_fields = len(SchemaFactory()._parse_schema_string(standard))
OutputSchema = SchemaFactory().create_from_standard(
    standard,
    record_class_name="WopkeRecord",
    output_class_name="WopkeOutput",
    records_key="yield_records",
)

provider_label = re.sub(r"[^A-Za-z0-9]+", "-", PROVIDER.strip()).strip("-")
model_label = re.sub(r"[^A-Za-z0-9]+", "-", MODEL_NAME.strip()).strip("-")
file_tag = f"{provider_label}_{model_label}_{n_fields}fields"

paper_input = Path(PAPER_INPUT_DIR)
if not paper_input.is_absolute():
    paper_input = repo_root / paper_input
if not paper_input.is_dir():
    raise FileNotFoundError(f"Paper input dir not found: {paper_input}")

run_dir = Path(DEFAULT_OUTPUT_DIR) / file_tag
by_paper_dir = run_dir / "by_paper"
by_paper_dir.mkdir(parents=True, exist_ok=True)
status_path = run_dir / "run_status.csv"


def make_paper_id(folder_name: str, study_id: int | None = None) -> str:
    m = re.match(r"^(\d+)\.\s*(.*)$", folder_name)
    rest = m.group(2) if m else folder_name
    slug = re.sub(r"[^A-Za-z0-9]+", "_", rest).strip("_")[:50]
    if study_id is not None:
        return f"{int(study_id):03d}_{slug}"
    if m:
        return f"{int(m.group(1)):03d}_{slug}"
    return slug[:80]


papers = []
for md_path in get_all_markdown_paths(base_dir=str(paper_input)):
    info = get_paper_info_from_path(md_path)
    folder = info["paper_folder"]
    study_id = study_id_for_folder(folder)
    if ONLY_MAPPED and study_id is None:
        continue
    papers.append(
        {
            "paper_id": make_paper_id(folder, study_id),
            "paper_folder": folder,
            "path": md_path,
            "study_id": study_id,
            "study_ids": [study_id] if study_id is not None else [],
        }
    )

if STUDY_IDS is not None:
    wanted = {int(s) for s in STUDY_IDS}
    papers = [p for p in papers if p["study_id"] in wanted]

skip_ids = {int(s) for s in (SKIP_STUDY_IDS or [])}
if skip_ids:
    before = len(papers)
    papers = [p for p in papers if p["study_id"] not in skip_ids]
    print(f"Skipped Study# {sorted(skip_ids)} ({before - len(papers)} papers)")

# Study# ascending (1→90); unfinished methods are picked up in METHODS order.
papers.sort(
    key=lambda p: (
        p["study_id"] is None,  # mapped first
        p["study_id"] if p["study_id"] is not None else 10**9,  # smallest Study# first
        p["paper_folder"],
    )
)


def method_done(paper_id: str, method: str) -> bool:
    path = by_paper_dir / f"{paper_id}__{method}.csv"
    return path.is_file() and path.stat().st_size > 0


def pending_methods(paper_id: str) -> list[str]:
    if not SKIP_EXISTING:
        return list(METHODS)
    return [m for m in METHODS if not method_done(paper_id, m)]


def paper_has_any_result(paper_id: str) -> bool:
    return any(method_done(paper_id, m) for m in METHODS)


def later_paper_has_progress(idx: int) -> bool:
    """True if any later Study# already has at least one method CSV."""
    for p in papers[idx + 1 :]:
        if paper_has_any_result(p["paper_id"]):
            return True
    return False


# Inventory existing by_paper outputs for resume.
# Incomplete papers behind a later finished paper are treated as abandoned
# (failed/skipped last time) and are not retried.
n_method_done = {m: 0 for m in METHODS}
n_paper_complete = 0
n_paper_partial = 0
n_paper_todo = 0
n_paper_gap_skip = 0
first_resume = None
for idx, p in enumerate(papers):
    pending = pending_methods(p["paper_id"])
    done_here = [m for m in METHODS if m not in pending]
    for m in done_here:
        n_method_done[m] += 1
    if not pending:
        n_paper_complete += 1
        continue
    if SKIP_EXISTING and later_paper_has_progress(idx):
        n_paper_gap_skip += 1
        continue
    if len(pending) < len(METHODS):
        n_paper_partial += 1
    else:
        n_paper_todo += 1
    if first_resume is None:
        first_resume = (p, pending)

n_mapped = sum(1 for p in papers if p["study_id"] is not None)
print(f"LLM      : {PROVIDER}/{MODEL_NAME}")
print(f"Standard : {STANDARD_KEY} ({n_fields} fields)")
print(f"Methods  : {METHODS}")
print(f"Input    : {paper_input}")
print(f"Papermap : {len(FOLDER_TO_STUDY_ID)} folders → Study# (unmapped folders: {len(UNMAPPED_FOLDERS)})")
print(f"Papers   : {len(papers)} (ONLY_MAPPED={ONLY_MAPPED})")
print(f"Mapped   : {n_mapped}/{len(papers)} have a GT Study#")
print(f"Output   : {run_dir}")
print(f"Tag      : {file_tag}")
print(f"Order    : Study# ascending (1→90)")
print(f"Resume   : SKIP_EXISTING={SKIP_EXISTING}")
print(
    f"Progress : complete={n_paper_complete} partial={n_paper_partial} "
    f"todo={n_paper_todo} gap_skip={n_paper_gap_skip} "
    f"| per-method done={dict(n_method_done)}"
)
if first_resume is not None:
    p0, pending0 = first_resume
    print(
        f"Next     : Study {p0['study_id']} · {p0['paper_id']} "
        f"→ pending {pending0}"
    )
else:
    print("Next     : nothing left (all selected papers × methods done)")
print(f"Logs     : SHOW_LOGS={SHOW_LOGS} QUIET_HTTP={QUIET_HTTP}")
log.info("Setup complete — ready to run %d papers", len(papers))

In [ ]:
mas_objective = f"""You are an expert agricultural meta-analysis specialist.
Your task is to extract **intercropping experiment records** from a scientific research paper.
This meta-analysis compares crop yield under intercropping settings versus sole cropping.

**META-ANALYTIC SCHEMA (CRITICAL)**:
{standard}

**SCHEMA RULES**
- Use JSON keys as the EXACT field names in every record.
- Schema descriptions are guidance only — values must be concrete text extracted from the paper.
- Do not rename, add, or remove any schema fields.

**MULTI-RECORD RULE (CRITICAL)**
Each unique combination of crop pair × site × year × treatment level = one SEPARATE record.
Each row in a yield results table is typically a separate record.
Do NOT collapse table rows or merge treatment combinations into a single record.

**YIELD FIELD MAPPING (CRITICAL)**
- `unified yield sc 1` = sole-crop yield of Crop species 1
- `unified yield sc 2` = sole-crop yield of Crop species 2
- `unified yield ic 1` = intercropped yield of Crop species 1
- `unified yield ic 2` = intercropped yield of Crop species 2
Preserve numeric values exactly — do not round or average.

**OUTPUT**: One record per unique treatment combination using exact schema field names.
"""

def paper_csv_path(paper_id: str, method: str) -> Path:
    return by_paper_dir / f"{paper_id}__{method}.csv"


def output_exists(paper_id: str, method: str) -> bool:
    path = paper_csv_path(paper_id, method)
    return path.is_file() and path.stat().st_size > 0


def records_to_df(results) -> pd.DataFrame:
    if hasattr(results, "model_dump"):
        payload = results.model_dump()
    elif hasattr(results, "dict"):
        payload = results.dict()
    elif isinstance(results, dict):
        payload = results
    else:
        raise TypeError(f"Unsupported results type: {type(results)}")
    records = payload.get("yield_records") or payload.get("records") or []
    if not isinstance(records, list):
        records = [records]
    rows_out = []
    for rec in records:
        if hasattr(rec, "model_dump"):
            rows_out.append(rec.model_dump())
        elif isinstance(rec, dict):
            rows_out.append(rec)
    return pd.DataFrame(rows_out)


def save_records(results, paper: dict, method: str) -> tuple[str, int]:
    df = records_to_df(results)
    df.insert(0, "paper_folder", paper["paper_folder"])
    df.insert(0, "method", method)
    df.insert(0, "study_ids", ";".join(str(s) for s in paper["study_ids"]))
    df.insert(0, "study_id", paper["study_id"] if paper["study_id"] is not None else "")
    df.insert(0, "paper_id", paper["paper_id"])
    path = paper_csv_path(paper["paper_id"], method)
    tmp = path.with_suffix(".csv.tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)
    return str(path), len(df)


def rebuild_combined(method: str) -> str | None:
    parts = sorted(by_paper_dir.glob(f"*__{method}.csv"))
    if not parts:
        return None
    frames = [pd.read_csv(p) for p in parts]
    out = run_dir / f"{method}.csv"
    pd.concat(frames, ignore_index=True).to_csv(out, index=False)
    return str(out)


def write_status(rows: list[dict]) -> None:
    pd.DataFrame(rows).to_csv(status_path, index=False)


def parse_mas_records(result_mas):
    if result_mas is None:
        return None
    workspace = getattr(result_mas, "final_workspace", None) or {}
    raw = workspace.get("final_meta_analysis_records", {})
    if isinstance(raw, OutputSchema):
        return raw
    if isinstance(raw, dict):
        return OutputSchema.model_validate(raw)
    return None


def run_direct(paper_path: str):
    return extract_meta_analysis(
        paper_path,
        schema=standard,
        provider=PROVIDER,
        model_name=MODEL_NAME,
        debug_raw_response=False,
    )


def run_workflow(paper_path: str):
    text = highlight_numbers_and_tables(read_paper_text(paper_path))
    out = run_two_step_text_to_dataset(
        text=text,
        workflow="label_then_direct",
        dataset_standard=standard,
        dataset_records_key="yield_records",
        record_class_name="WopkeRecord",
        output_class_name="WopkeOutput",
        label_step2_prompt_style="direct_full",
        label_step2_include_tag_note=True,
        label_step2_maximize_completeness=True,
        labeled_text_max_chars=120_000,
        provider=PROVIDER,
        model_name=MODEL_NAME,
    )
    return out.get("schema_output")


def run_mas(paper_path: str, paper_id: str):
    context = create_context(source=paper_path, name=f"paper_{paper_id}")
    orchestrator = Orchestrator(
        topology_name=MAS_TOPOLOGY,
        provider=PROVIDER,
        model_name=MODEL_NAME,
    )
    result = orchestrator.run(
        source=context,
        objective=mas_objective,
        output_schema=OutputSchema,
    )
    return parse_mas_records(result)

In [ ]:
import time
from datetime import datetime

runners = {
    "direct_llm": lambda path, paper: run_direct(path),
    "static_workflow": lambda path, paper: run_workflow(path),
    "mas": lambda path, paper: run_mas(path, paper["paper_id"]),
}

for method in METHODS:
    rebuild_combined(method)

rows = []
done_ok = done_skip = done_fail = 0
t0_all = time.perf_counter()


def progress(msg: str) -> None:
    line = f"{datetime.now().strftime('%H:%M:%S')} | {msg}"
    tqdm.write(line)
    log.info(msg)


n_jobs = 0
for idx, p in enumerate(papers):
    pending = pending_methods(p["paper_id"])
    if not pending:
        continue
    if SKIP_EXISTING and later_paper_has_progress(idx):
        continue
    n_jobs += len(pending)

progress(
    f"Starting run: {len(papers)} papers × {METHODS} "
    f"({n_jobs} remaining jobs) → {run_dir}"
)

pbar = tqdm(papers, desc="Papers", unit="paper")
for i, paper in enumerate(pbar, start=1):
    paper_id = paper["paper_id"]
    paper_path = paper["path"]
    idx = i - 1
    pending = pending_methods(paper_id)
    if SKIP_EXISTING and not pending:
        # All 3 experiments already done — move to next Study#.
        done_skip += len(METHODS)
        for method in METHODS:
            n_rec = None
            try:
                n_rec = max(len(pd.read_csv(paper_csv_path(paper_id, method))), 0)
            except Exception:
                pass
            rows.append(
                {
                    "paper_id": paper_id,
                    "study_id": paper["study_id"],
                    "method": method,
                    "status": "skipped",
                    "n_records": n_rec,
                    "path": str(paper_csv_path(paper_id, method)),
                }
            )
        write_status(rows)
        progress(
            f"[{i}/{len(papers)}] Study {paper['study_id']} · {paper_id[:50]} "
            f"— all {len(METHODS)} methods done, next"
        )
        continue

    # Incomplete, but a later paper already has results → abandoned last run; don't retry.
    if SKIP_EXISTING and later_paper_has_progress(idx):
        done_skip += len(pending)
        for method in pending:
            rows.append(
                {
                    "paper_id": paper_id,
                    "study_id": paper["study_id"],
                    "method": method,
                    "status": "skipped_gap",
                    "n_records": 0,
                    "path": "",
                }
            )
        write_status(rows)
        progress(
            f"[{i}/{len(papers)}] Study {paper['study_id']} · {paper_id[:50]} "
            f"— incomplete {pending}, but later paper has results → skip gap, next"
        )
        continue

    progress(
        f"[{i}/{len(papers)}] Study {paper['study_id']} · {paper_id[:50]} "
        f"— pending {pending}"
    )
    for method in METHODS:
        pbar.set_postfix_str(f"{paper_id[:24]} · {method}", refresh=True)
        if SKIP_EXISTING and output_exists(paper_id, method):
            n_rec = None
            try:
                n_rec = max(len(pd.read_csv(paper_csv_path(paper_id, method))), 0)
            except Exception:
                pass
            rows.append(
                {
                    "paper_id": paper_id,
                    "study_id": paper["study_id"],
                    "method": method,
                    "status": "skipped",
                    "n_records": n_rec,
                    "path": str(paper_csv_path(paper_id, method)),
                }
            )
            write_status(rows)
            done_skip += 1
            progress(f"  skip  {method} (existing, n_records={n_rec})")
            continue
        progress(f"  start {method} …")
        t0 = time.perf_counter()
        try:
            result = runners[method](paper_path, paper)
            elapsed = time.perf_counter() - t0
            if result is None:
                rows.append(
                    {
                        "paper_id": paper_id,
                        "study_id": paper["study_id"],
                        "method": method,
                        "status": "no_output",
                        "n_records": 0,
                        "path": "",
                    }
                )
                write_status(rows)
                done_fail += 1
                progress(f"  fail  {method} (no_output) in {elapsed:.1f}s")
                continue
            path, n_rec = save_records(result, paper, method)
            rebuild_combined(method)
            rows.append(
                {
                    "paper_id": paper_id,
                    "study_id": paper["study_id"],
                    "method": method,
                    "status": "ok",
                    "n_records": n_rec,
                    "path": path,
                }
            )
            write_status(rows)
            done_ok += 1
            progress(f"  ok    {method}: {n_rec} records in {elapsed:.1f}s → {Path(path).name}")
        except Exception as exc:
            elapsed = time.perf_counter() - t0
            rows.append(
                {
                    "paper_id": paper_id,
                    "study_id": paper["study_id"],
                    "method": method,
                    "status": f"error: {type(exc).__name__}: {exc}",
                    "n_records": 0,
                    "path": "",
                }
            )
            write_status(rows)
            done_fail += 1
            progress(f"  error {method} after {elapsed:.1f}s: {type(exc).__name__}: {exc}")

for method in METHODS:
    rebuild_combined(method)

elapsed_all = time.perf_counter() - t0_all
progress(
    f"Finished in {elapsed_all/60:.1f} min — ok={done_ok} skipped={done_skip} failed={done_fail}"
)

summary = pd.DataFrame(rows)
summary

In [ ]:
if summary.empty:
    print("No runs.")
else:
    print(f"Status file: {status_path}")
    print(summary.groupby(["method", "status"]).size().unstack(fill_value=0))
    failed = summary[
        summary["status"].astype(str).str.startswith("error") | summary["status"].eq("no_output")
    ]
    if failed.empty:
        print("No failures.")
    else:
        print(f"\nFailed ({len(failed)}). Re-run this notebook to retry them.")
failed if not summary.empty else summary